# NutriMatch - Orang B: Allergen & Ingredient Preprocessing

Notebook ini mengerjakan bagian Orang B: membersihkan dataset alergen, membuat kamus keyword, membuat label multi-label, menandai confidence, dan menghasilkan output siap handover ke AI Engineer.

Output utama ada di `outputs/orang_b`:

- `allergen_reference_clean.csv`
- `food_allergen_labels.csv`
- `ingredient_allergen_training.csv`
- `allergen_data_quality_report.csv`
- `allergen_eda_summary.csv`
- `handover_notes_orang_b.md`

Catatan Open Food Facts: file lokal sangat besar, jadi pipeline default membaca secara chunk dan mengambil maksimal 50.000 record yang punya sinyal alergen/ingredient. Untuk scan lebih luas, naikkan `OFF_MAX_CHUNKS` atau ubah menjadi `None`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'Dataset'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'orang_b'
SRC_DIR = PROJECT_ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 120)

print('Project root:', PROJECT_ROOT)
print('Data dir:', DATA_DIR)
print('Output dir:', OUTPUT_DIR)

## 1. Audit Dataset

Cell ini membaca dataset kecil/menengah secara penuh dan Open Food Facts hanya header/sampel awal karena ukurannya besar.

In [ ]:
csv_files = sorted(DATA_DIR.glob('*.csv'))
audit_rows = []

for path in csv_files:
    size_mb = path.stat().st_size / 1024 / 1024
    try:
        if path.name == 'en.openfoodfacts.org.products.csv':
            sample = pd.read_csv(path, sep='\t', nrows=3, low_memory=False)
            shape = ('large', len(sample.columns))
            duplicates = None
            missing_total = None
        else:
            df = pd.read_csv(path)
            shape = df.shape
            duplicates = int(df.duplicated().sum())
            missing_total = int(df.isna().sum().sum())
            sample = df.head(3)
        audit_rows.append({
            'file': path.name,
            'size_mb': round(size_mb, 2),
            'shape': shape,
            'duplicates': duplicates,
            'missing_total': missing_total,
            'columns': ', '.join(sample.columns[:12]) + (' ...' if len(sample.columns) > 12 else '')
        })
    except Exception as exc:
        audit_rows.append({'file': path.name, 'size_mb': round(size_mb, 2), 'shape': 'ERROR', 'duplicates': None, 'missing_total': None, 'columns': str(exc)})

audit_df = pd.DataFrame(audit_rows)
audit_df

## 2. Jalankan Pipeline Orang B

Default aman untuk laptop:

- `OFF_MAX_CHUNKS = 20`: baca 20 chunk Open Food Facts.
- `OFF_CHUNK_SIZE = 100_000`: setiap chunk 100 ribu baris.
- `OFF_MAX_RECORDS = 50_000`: ambil maksimal 50 ribu record Open Food Facts hasil filter.

Jika ingin full scan Open Food Facts, ubah `OFF_MAX_CHUNKS = None`, tapi runtime dan ukuran output akan jauh lebih besar.

In [ ]:
from orang_b_allergen_pipeline import run_pipeline

OFF_MAX_CHUNKS = 20
OFF_CHUNK_SIZE = 100_000
OFF_MAX_RECORDS = 50_000

paths = run_pipeline(
    off_max_chunks=OFF_MAX_CHUNKS,
    off_chunk_size=OFF_CHUNK_SIZE,
    off_max_records=OFF_MAX_RECORDS,
)

paths

## 3. Cek Output Utama

In [ ]:
labels = pd.read_csv(OUTPUT_DIR / 'food_allergen_labels.csv')
reference = pd.read_csv(OUTPUT_DIR / 'allergen_reference_clean.csv')
quality = pd.read_csv(OUTPUT_DIR / 'allergen_data_quality_report.csv')
eda = pd.read_csv(OUTPUT_DIR / 'allergen_eda_summary.csv')

print('labels:', labels.shape)
print('reference:', reference.shape)
print('quality:', quality.shape)
display(labels.head())
display(quality)

## 4. Distribusi Label Alergen

In [ ]:
allergen_cols = [col for col in labels.columns if col.startswith('contains_')]
label_counts = labels[allergen_cols].sum().sort_values(ascending=True)

ax = label_counts.plot(kind='barh', figsize=(10, 6), color='#4C78A8')
ax.set_title('Distribusi Label Alergen')
ax.set_xlabel('Jumlah baris')
ax.set_ylabel('Label')
plt.tight_layout()
plt.show()

label_counts.sort_values(ascending=False).to_frame('count')

## 5. Confidence dan Sumber Label

Interpretasi:

- `high`: berasal dari allergen tag, boolean allergen, atau empty list eksplisit.
- `medium`: berasal dari keyword match ingredient/product text.
- `low`: umumnya nama makanan saja atau unknown, perlu review manual bila masuk makanan utama.

In [ ]:
display(labels['confidence'].value_counts().to_frame('rows'))
display(pd.crosstab(labels['source'], labels['confidence']))
display(labels['label_source'].value_counts().head(20).to_frame('rows'))

## 6. Manual Review Queue

Baris `contains_unknown = True` sebaiknya dicek manual kalau makanan tersebut akan masuk rekomendasi. Untuk aplikasi alergi, unknown lebih aman diperlakukan sebagai tidak aman pada mode conservative.

In [ ]:
manual_review = labels[
    labels['contains_unknown'].eq(True)
    | labels['confidence'].eq('low')
].copy()

review_cols = ['source', 'product_name', 'ingredient_text', 'allergens_raw', 'label_source', 'confidence', 'contains_unknown']
manual_review[review_cols].head(30)

## 7. File untuk Handover ke AI Engineer

Gunakan `food_allergen_labels.csv` sebagai label multi-label utama. Untuk eksperimen klasifikasi teks ingredient, gunakan `ingredient_allergen_training.csv`.

In [ ]:
training = pd.read_csv(OUTPUT_DIR / 'ingredient_allergen_training.csv')
print('Training subset:', training.shape)
display(training[['source', 'product_name', 'ingredient_text', 'allergens_raw', 'confidence'] + allergen_cols].head())

print('\nOutput files:')
for path in sorted(OUTPUT_DIR.glob('*')):
    print('-', path.name)